# Phase 3 - Bias-Variance Tradeoff

**Team IDs:** 16002932, 16004208, 16006616  |  **Seed:** 42648

**Rules:** gradient descent from `gradient_descent.py` for the fitted models; closed-form least squares is used only for the cross-validation / chronological *screening* of every rung (labelled where it appears); `test.csv` is never used.

## Expectation

In [1]:
# -- Imports and constants ---------------------------------------------------
import sys, time, hashlib, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GroupKFold

import gradient_descent as core
import polynomial_regression as p2
import bias_variance as p3

warnings.filterwarnings('ignore')
np.set_printoptions(precision=6, suppress=True)

SEED = 42648
IDS = ['16002932', '16006616', '16004208']
P1_STEM, P2_STEM, P3_STEM = 'gradient_descent_artifacts', 'polynomial_regression_artifacts', 'bias_variance_artifacts'
PLOTS = Path('plots'); PLOTS.mkdir(exist_ok=True)
T0 = time.time()
print(f'Python {sys.version.split()[0]}  numpy {np.__version__}  pandas {pd.__version__}')

Python 3.14.6  numpy 2.5.0  pandas 3.0.3


## 1. Provenance: consume Phase 1 and Phase 2

In [2]:
W_P1, B_P1, META_P1, p1_train_idx, p1_val_idx = core.load_artifacts(P1_STEM)
W_P2, B_P2, META_P2 = p2.load_artifacts(P2_STEM)

assert hashlib.sha256(W_P1.tobytes()).hexdigest() == META_P1['w_sha256']
assert hashlib.sha256(W_P2.tobytes()).hexdigest() == META_P2['phase2_w_sha256']
assert META_P2['phase1_w_sha256'] == META_P1['w_sha256']
assert core.compute_seed(IDS) == SEED == META_P2['seed']
print('[PASS] Phase 1 -> Phase 2 hash chain and seed')

train_all = pd.read_csv('Datasets/train.csv')
train_set, val_set = train_test_split(train_all, test_size=0.20, random_state=SEED)
assert np.array_equal(train_set.index.values, p1_train_idx) and np.array_equal(val_set.index.values, p1_val_idx)
print(f'[PASS] seeded split verified: train={len(train_set)}, val={len(val_set)}')

P2_DEGREE = META_P2['chosen_degree']
P2_SOURCES = META_P2['source_cols']
P2_FEATS = META_P2['expanded_feature_names']
print(f'Phase 2 chose degree {P2_DEGREE} on {P2_SOURCES} ({len(P2_FEATS)} features), val R2 {META_P2["val_r2"]:.6f}')

# Phase 1 preprocessing refit on the training part must equal the stored one
pre0 = core.fit_preprocessor(train_set)
for k in pre0['scaler_means']:
    assert abs(pre0['scaler_means'][k] - META_P1['scaler_means'][k]) < 1e-12
print('[PASS] preprocessor fitted on the training part equals the stored Phase 1 parameters')

[PASS] Phase 1 -> Phase 2 hash chain and seed
[PASS] seeded split verified: train=8708, val=2178
Phase 2 chose degree 3 on ['temp', 'hum', 'windspeed', 'days_since_start'] (62 features), val R2 0.685827
[PASS] preprocessor fitted on the training part equals the stored Phase 1 parameters


## 2. Helpers: design matrices, closed-form screening, metrics

In [3]:
WEATHER = [c for c in P2_SOURCES if c != 'days_since_start']     # the trend column is examined on its own below
yt = train_set['cnt'].values.astype(float)
yv = val_set['cnt'].values.astype(float)


def design(fit_df, other_dfs, spec):
    """Phase 1 preprocessing + Phase 3 expansion, parameters fitted on fit_df only."""
    pm = core.fit_preprocessor(fit_df)
    X1, n1 = core.transform(fit_df, pm)
    pp = p3.fit_poly(X1, n1, spec)
    outs = []
    for df in [fit_df] + list(other_dfs):
        X, _ = core.transform(df, pm)
        Xe, names = p3.transform_poly(X, n1, pp)
        outs.append(Xe)
    return outs, names, pm, pp


def ols(X, y):
    A = np.hstack([X, np.ones((len(X), 1))])
    th = np.linalg.lstsq(A, y, rcond=None)[0]
    return th[:-1], float(th[-1])


def screen_r2(spec, fit_df, ev_df):
    """Closed-form screening: R2 on ev_df of a model fitted on fit_df (all preprocessing refit on fit_df)."""
    (Xf, Xe), _, _, _ = design(fit_df, [ev_df], spec)
    w, b = ols(Xf, fit_df['cnt'].values.astype(float))
    return core.r2_score_np(ev_df['cnt'].values.astype(float), core.predict(Xe, w, b))


def screen_cv(spec, df, n_splits=5):
    """Day-grouped K-fold (whole days stay together) inside df."""
    r = [screen_r2(spec, df.iloc[ia], df.iloc[iv])
         for ia, iv in GroupKFold(n_splits=n_splits).split(df, groups=df['dteday'].values)]
    return float(np.mean(r)), float(np.std(r))


# One chronological cut for the whole project: sort by date and hour, first 80% trains
chrono_all = train_all.sort_values(['dteday', 'hr']).reset_index(drop=True)
cut = int(len(chrono_all) * 0.8)
chrono_tr, chrono_va = chrono_all.iloc[:cut], chrono_all.iloc[cut:]
print(f'chronological cut after {chrono_tr["dteday"].max()}: train {len(chrono_tr)}, validation {len(chrono_va)}')

chronological cut after 2012-08-05: train 8708, validation 2178


## 3. Diagnosis rule (fixed before the ladder is run)

We add model capacity one step at a time. After every step we record the train R2, the seeded validation R2, the day-grouped cross-validation R2 (computed **inside the training part only**, so the validation set never influences the choice) and the chronological R2.

* **High bias (under-fit):** the next step still raises the CV R2 by more than 0.005 while the train - validation gap stays below 0.02.
* **High variance (over-fit):** the train - validation gap exceeds 0.04, or the CV / chronological R2 *falls* when capacity is added.
* **Balanced:** CV R2 within 0.005 of the best. We carry forward the **least complex** such model (parsimony).

## 4. The complexity ladder

The first rungs are Phase 1 (linear) and Phase 2 (polynomial powers, 4 source columns, degree 3). Rung R2 keeps the degree Phase 2 chose but expands only the weather columns, because the trend column's powers are tested separately (R1 vs R2). Every later rung adds one family of interaction terms. Phase 1 and Phase 2 rungs use the stored weights (their validation scores must reproduce); every other rung is fitted with our own gradient descent, warm-started **by feature name** from the rung before.

In [4]:
POW_P2 = {c: P2_DEGREE for c in P2_SOURCES}
POW_W = {c: P2_DEGREE for c in WEATHER}
STEPS = [
    ('R3 + hr x workingday',                    [('hr', 'workingday')]),
    ('R4 + hr x weather',                       [('hr', 'weather')]),
    ('R5 + hr x weekday',                       [('hr', 'weekday')]),
    ('R6 + hr x mnth',                          [('hr', 'mnth')]),
    ('R7 + hr x temp',                          [('hr', 'temp')]),
    ('R8 + hr x hum',                           [('hr', 'hum')]),
    ('R9 + hr x yr',                            [('hr', 'yr')]),
    ('R10 + workingday x yr, workingday x mnth', [('workingday', 'yr'), ('workingday', 'mnth')]),
    ('R11 + hr x windspeed, hr x season, hr x holiday', [('hr', 'windspeed'), ('hr', 'season'), ('hr', 'holiday')]),
]
LADDER = [('R0 linear (Phase 1)', {'degrees': {}}),
          ('R1 Phase 2 model (powers of temp, hum, windspeed, days_since_start)', {'degrees': POW_P2}),
          ('R2 powers of temp, hum, windspeed only', {'degrees': POW_W})]
inter = []
for nm, add in STEPS:
    inter = inter + [list(t) for t in add]
    LADDER.append((nm, {'degrees': POW_W, 'interactions': [list(t) for t in inter]}))
for nm, sp in LADDER: print(nm)

R0 linear (Phase 1)
R1 Phase 2 model (powers of temp, hum, windspeed, days_since_start)
R2 powers of temp, hum, windspeed only
R3 + hr x workingday
R4 + hr x weather
R5 + hr x weekday
R6 + hr x mnth
R7 + hr x temp
R8 + hr x hum
R9 + hr x yr
R10 + workingday x yr, workingday x mnth
R11 + hr x windspeed, hr x season, hr x holiday


In [5]:
rows, fitted = [], {}
prev = None   # (weights, names) of the previous rung on the main path
t_ladder = time.time()
for k, (nm, spec) in enumerate(LADDER):
    if k == 1:
        # Phase 2's own expansion (its column order differs from Phase 3's), so that the stored weights apply exactly
        pm1 = core.fit_preprocessor(train_set)
        X1a, n1_ = core.transform(train_set, pm1); X1v, _ = core.transform(val_set, pm1)
        pp = p2.fit_poly(X1a, n1_, P2_SOURCES, P2_DEGREE, False)
        Xa, names = p2.transform_poly(X1a, n1_, pp); Xv, _ = p2.transform_poly(X1v, n1_, pp)
    else:
        (Xa, Xv), names, pm, pp = design(train_set, [val_set], spec)
    p = Xa.shape[1]
    if k == 0:
        w, b, status, iters, kappa = W_P1, B_P1, 'stored', 0, np.nan
        assert len(w) == p
    elif k == 1:
        w, b, status, iters, kappa = W_P2, B_P2, 'stored', 0, np.nan
        assert names == P2_FEATS and len(w) == p
    else:
        w_prev, names_prev = prev
        w0 = p3.warm_start_by_name(w_prev, names_prev, names)
        L = core.lipschitz_constant(Xa)
        w, b, kappa, res = p3.fit_expanded(Xa, yt, Xv, yv, w_init=w0, b_init=prev_b, lr=0.9 / L, tol=1e-7, max_iter=300000)
        status, iters = res['status'], res['n_iter']
        assert status == 'converged', (nm, status)
    tr_r2 = core.r2_score_np(yt, core.predict(Xa, w, b)); va_r2 = core.r2_score_np(yv, core.predict(Xv, w, b))
    va_rmse = core.rmse_np(yv, core.predict(Xv, w, b))
    cv_m, cv_s = screen_cv(spec, train_set)
    ch_r2 = screen_r2(spec, chrono_tr, chrono_va)
    rows.append(dict(rung=nm, p=int(p), train_r2=tr_r2, val_r2=va_r2, gap=tr_r2 - va_r2, val_rmse=va_rmse,
                     cv_r2=cv_m, cv_std=cv_s, chrono_r2=ch_r2, status=status, iters=int(iters), kappa=float(kappa)))
    fitted[nm] = dict(w=w, b=b, names=names, pp=pp, spec=spec)
    if k != 1:                                  # R1 is a side branch (it contains powers of the trend column)
        prev, prev_b = (w, names), b
    r = rows[-1]
    print(f'{nm[:58]:<58} p={r["p"]:>4}  train {tr_r2:.4f}  val {va_r2:.4f}  CV {cv_m:.4f}+-{cv_s:.4f}  chrono {ch_r2:.4f}  [{status}, {iters} it]', flush=True)
    if k == 0:
        prev, prev_b = (w, names), b
print(f'ladder took {time.time()-t_ladder:.0f}s')

# chain checks: the stored Phase 1 / Phase 2 weights reproduce their recorded scores on this split
assert abs(rows[1]['val_r2'] - META_P2['val_r2']) < 1e-9
print(f'[PASS] R1 reproduces the Phase 2 validation R2 ({rows[1]["val_r2"]:.6f})')
ladder_df = pd.DataFrame(rows)

R0 linear (Phase 1)                                        p=  54  train 0.6982  val 0.6781  CV 0.6923+-0.0132  chrono 0.6364  [stored, 0 it]


R1 Phase 2 model (powers of temp, hum, windspeed, days_sin p=  62  train 0.7044  val 0.6858  CV 0.6969+-0.0103  chrono 0.5099  [stored, 0 it]


R2 powers of temp, hum, windspeed only                     p=  60  train 0.7045  val 0.6858  CV 0.6970+-0.0103  chrono 0.6406  [converged, 1614 it]


R3 + hr x workingday                                       p=  83  train 0.8372  val 0.8445  CV 0.8287+-0.0101  chrono 0.8035  [converged, 8427 it]


R4 + hr x weather                                          p= 129  train 0.8452  val 0.8509  CV 0.8346+-0.0069  chrono 0.8104  [converged, 2837 it]


...

R8 + hr x hum                                              p= 566  train 0.8991  val 0.8834  CV 0.8797+-0.0087  chrono 0.8241  [converged, 4049 it]


R9 + hr x yr                                               p= 589  train 0.9272  val 0.9107  CV 0.9125+-0.0058  chrono 0.9063  [converged, 13924 it]


R10 + workingday x yr, workingday x mnth                   p= 601  train 0.9283  val 0.9109  CV 0.9128+-0.0054  chrono 0.9061  [converged, 3903 it]


R11 + hr x windspeed, hr x season, hr x holiday            p= 716  train 0.9286  val 0.9109  CV 0.9127+-0.0053  chrono 0.9054  [converged, 763 it]


ladder took 170s
[PASS] R1 reproduces the Phase 2 validation R2 (0.685827)


## 5. Evidence tables and plot

In [6]:
show = ladder_df[['rung', 'p', 'train_r2', 'val_r2', 'gap', 'cv_r2', 'cv_std', 'chrono_r2', 'iters']].copy()
show['rung'] = show['rung'].str.slice(0, 52)
print(show.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

main = ladder_df[~ladder_df['rung'].str.startswith('R1 ')]
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(main['p'], main['train_r2'], 'b-o', label='train (seeded)')
ax.plot(main['p'], main['val_r2'], 'r-s', label='validation (seeded)')
ax.plot(main['p'], main['cv_r2'], 'g-^', label='day-grouped CV (train part)')
ax.plot(main['p'], main['chrono_r2'], 'm-d', label='chronological')
r1 = ladder_df[ladder_df['rung'].str.startswith('R1 ')].iloc[0]
ax.scatter([r1['p']], [r1['chrono_r2']], c='k', marker='x', s=80, label='R1 chronological (trend powers)')
ax.set_xscale('log'); ax.set_xlabel('number of features (model complexity)'); ax.set_ylabel('R2')
ax.set_title('Phase 3: complexity ladder'); ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); fig.savefig(PLOTS / '03_complexity_ladder.png', dpi=140); plt.close(fig)
print('[SAVED] plots/03_complexity_ladder.png')

                                                rung   p  train_r2  val_r2     gap  cv_r2  cv_std  chrono_r2  iters
                                 R0 linear (Phase 1)  54    0.6982  0.6781  0.0200 0.6923  0.0132     0.6364      0
R1 Phase 2 model (powers of temp, hum, windspeed, da  62    0.7044  0.6858  0.0186 0.6969  0.0103     0.5099      0
              R2 powers of temp, hum, windspeed only  60    0.7045  0.6858  0.0187 0.6970  0.0103     0.6406   1614
                                R3 + hr x workingday  83    0.8372  0.8445 -0.0072 0.8287  0.0101     0.8035   8427
                                   R4 + hr x weather 129    0.8452  0.8509 -0.0056 0.8346  0.0069     0.8104   2837
                                   R5 + hr x weekday 267    0.8539  0.8500  0.0039 0.8379  0.0080     0.8172   2049
                                      R6 + hr x mnth 520    0.8880  0.8739  0.0141 0.8688  0.0060     0.8120  13427
                                      R7 + hr x temp 543    0.8944  0.88

[SAVED] plots/03_complexity_ladder.png


## 6. Trend-column powers: Phase 2 vs weather-only (R1 vs R2)

Phase 2 expanded `days_since_start` too. Powers of a time index are harmless inside the training period and explode outside it, so this is tested directly.

In [7]:
a1, a2 = ladder_df.iloc[1], ladder_df.iloc[2]
print(f'{"":<46}{"p":>5}{"val R2":>9}{"CV R2":>9}{"chrono R2":>11}')
for r in (a1, a2):
    print(f'{r["rung"][:44]:<46}{r["p"]:>5}{r["val_r2"]:>9.4f}{r["cv_r2"]:>9.4f}{r["chrono_r2"]:>11.4f}')
print(f'\nremoving the powers of days_since_start changes: val {a2["val_r2"]-a1["val_r2"]:+.4f}, CV {a2["cv_r2"]-a1["cv_r2"]:+.4f}, chronological {a2["chrono_r2"]-a1["chrono_r2"]:+.4f}')

# same comparison once the interactions are in (R9 with and without the trend powers)
R9 = next(s for n, s in LADDER if n.startswith('R9 '))
spec_R9_trend = {'degrees': POW_P2, 'interactions': R9['interactions']}
cv9t = screen_cv(spec_R9_trend, train_set)[0]; ch9t = screen_r2(spec_R9_trend, chrono_tr, chrono_va); va9t = screen_r2(spec_R9_trend, train_set, val_set)
r9 = ladder_df[ladder_df['rung'].str.startswith('R9 ')].iloc[0]
print(f'\nR9 interactions WITH trend powers   : val {va9t:.4f}  CV {cv9t:.4f}  chronological {ch9t:.4f}')
print(f'R9 interactions WITHOUT trend powers: val {r9["val_r2"]:.4f}  CV {r9["cv_r2"]:.4f}  chronological {r9["chrono_r2"]:.4f}')

                                                  p   val R2    CV R2  chrono R2
R1 Phase 2 model (powers of temp, hum, winds     62   0.6858   0.6969     0.5099
R2 powers of temp, hum, windspeed only           60   0.6858   0.6970     0.6406

removing the powers of days_since_start changes: val -0.0001, CV +0.0001, chronological +0.1307



R9 interactions WITH trend powers   : val 0.9107  CV 0.9132  chronological 0.8508
R9 interactions WITHOUT trend powers: val 0.9107  CV 0.9125  chronological 0.9063


## 7. Diagnosis and the complexity carried forward

In [8]:
TOL_SEL = 0.005
path = [r for r in rows if not r['rung'].startswith('R1 ')]          # main path: R0, R2, R3, ...
pick = p3.select_parsimonious_complexity(path, tol=TOL_SEL, score_key='cv_r2', size_key='p')
best_cv = max(r['cv_r2'] for r in path)
print(f'best CV R2 on the path: {best_cv:.4f}; rule: least complex rung with CV R2 >= {best_cv - TOL_SEL:.4f}')
print(f'CARRIED FORWARD: {pick["rung"]}  (p={pick["p"]}, CV {pick["cv_r2"]:.4f}, val {pick["val_r2"]:.4f}, chronological {pick["chrono_r2"]:.4f})\n')

print('step-by-step diagnosis (rule above):')
for a_, b_ in zip(path[:-1], path[1:]):
    d_cv = b_['cv_r2'] - a_['cv_r2']
    if b_['gap'] > 0.04 or d_cv < -TOL_SEL:
        tag = 'HIGH VARIANCE signal'
    elif d_cv > TOL_SEL and b_['gap'] < 0.02:
        tag = 'high bias (capacity still pays)'
    else:
        tag = 'plateau (within tolerance)'
    print(f'  {a_["rung"][:3].strip():<4}-> {b_["rung"][:3].strip():<4} dCV {d_cv:+.4f}  gap {b_["gap"]:+.4f}  {tag}')
print(f'\nlargest train - validation gap on the path: {max(r["gap"] for r in path):+.4f} (variance threshold 0.04)')
r1_, r2_ = rows[1], rows[2]
print(f'out-of-time variance signal: powers of the trend column change chronological R2 by {r2_["chrono_r2"]-r1_["chrono_r2"]:+.4f} while CV changes by {r2_["cv_r2"]-r1_["cv_r2"]:+.4f}')

best CV R2 on the path: 0.9128; rule: least complex rung with CV R2 >= 0.9078
CARRIED FORWARD: R9 + hr x yr  (p=589, CV 0.9125, val 0.9107, chronological 0.9063)

step-by-step diagnosis (rule above):
  R0  -> R2   dCV +0.0047  gap +0.0187  plateau (within tolerance)
  R2  -> R3   dCV +0.1317  gap -0.0072  high bias (capacity still pays)
  R3  -> R4   dCV +0.0059  gap -0.0056  high bias (capacity still pays)
  R4  -> R5   dCV +0.0033  gap +0.0039  plateau (within tolerance)
  R5  -> R6   dCV +0.0309  gap +0.0141  high bias (capacity still pays)
  R6  -> R7   dCV +0.0064  gap +0.0145  high bias (capacity still pays)
  R7  -> R8   dCV +0.0045  gap +0.0158  plateau (within tolerance)
  R8  -> R9   dCV +0.0328  gap +0.0165  high bias (capacity still pays)
  R9  -> R10  dCV +0.0003  gap +0.0173  plateau (within tolerance)
  R10 -> R11  dCV -0.0001  gap +0.0177  plateau (within tolerance)

largest train - validation gap on the path: +0.0200 (variance threshold 0.04)
out-of-time variance signa

In [9]:
chosen = fitted[pick['rung']]
SPEC = chosen['spec']
W_C, B_C, NAMES_C, PP_C = chosen['w'], chosen['b'], chosen['names'], chosen['pp']
print('carried-forward model:', pick['rung'])
print('features:', len(NAMES_C), '| spec degrees:', SPEC['degrees'])
print('interaction families:', SPEC.get('interactions', []))

# closed-form vs gradient descent at the carried-forward rung
(Xa_c, Xv_c), _, _, _ = design(train_set, [val_set], SPEC)
w_cf, b_cf = ols(Xa_c, yt)
print(f'val R2: gradient descent {core.r2_score_np(yv, core.predict(Xv_c, W_C, B_C)):.6f}   closed form {core.r2_score_np(yv, core.predict(Xv_c, w_cf, b_cf)):.6f}')

carried-forward model: R9 + hr x yr
features: 589 | spec degrees: {'temp': 3, 'hum': 3, 'windspeed': 3}
interaction families: [['hr', 'workingday'], ['hr', 'weather'], ['hr', 'weekday'], ['hr', 'mnth'], ['hr', 'temp'], ['hr', 'hum'], ['hr', 'yr']]


val R2: gradient descent 0.910669   closed form 0.910370


## 8. Chronological split: how much worse, and why (Req. of Phase 3)

The cut is the same for every rung (section 2). The test file (`test.csv`) is the 20th of every month, i.e. it lies *inside* the training period, so a "test-like" estimate holds out days 18-19 of every month as well.

In [10]:
hold = pd.to_datetime(train_all['dteday']).dt.day.isin([18, 19])
t_like_tr, t_like_va = train_all[~hold], train_all[hold]
print(f'test-like holdout: train {len(t_like_tr)} rows, held-out {len(t_like_va)} rows (days 18-19 of every month)\n')
print(f'{"model":<54}{"p":>5}{"seeded val":>12}{"CV":>8}{"chrono":>9}{"days 18-19":>12}')
test_like = {}
for r in rows:
    if r['rung'][:3].strip() in ('R0', 'R1', 'R2', 'R4') or r['rung'] == pick['rung']:
        tl = screen_r2(fitted[r['rung']]['spec'], t_like_tr, t_like_va)
        test_like[r['rung']] = tl
        print(f'{r["rung"][:52]:<54}{r["p"]:>5}{r["val_r2"]:>12.4f}{r["cv_r2"]:>8.4f}{r["chrono_r2"]:>9.4f}{tl:>12.4f}')

# gradient descent on the chronological split for the carried-forward model (the screening above is closed form)
(Xc_tr, Xc_va), _, _, _ = design(chrono_tr, [chrono_va], SPEC)
Lc = core.lipschitz_constant(Xc_tr)
w_ch, b_ch, _, res_ch = p3.fit_expanded(Xc_tr, chrono_tr['cnt'].values.astype(float), Xc_va, chrono_va['cnt'].values.astype(float),
                                        w_init=np.zeros(Xc_tr.shape[1]), b_init=0.0, lr=0.9 / Lc, tol=1e-7, max_iter=300000)
CH_GD = core.r2_score_np(chrono_va['cnt'].values.astype(float), core.predict(Xc_va, w_ch, b_ch))
print(f'\ncarried-forward model, chronological split by gradient descent: R2 {CH_GD:.4f} (status {res_ch["status"]}, {res_ch["n_iter"]} iterations); closed form {pick["chrono_r2"]:.4f}')
print(f'seeded validation {pick["val_r2"]:.4f}  vs  chronological {CH_GD:.4f}: gap {pick["val_r2"]-CH_GD:+.4f}')

test-like holdout: train 9749 rows, held-out 1137 rows (days 18-19 of every month)

model                                                     p  seeded val      CV   chrono  days 18-19
R0 linear (Phase 1)                                      54      0.6781  0.6923   0.6364      0.6911
R1 Phase 2 model (powers of temp, hum, windspeed, da     62      0.6858  0.6969   0.5099      0.6965
R2 powers of temp, hum, windspeed only                   60      0.6858  0.6970   0.6406      0.6967


R4 + hr x weather                                       129      0.8509  0.8346   0.8104      0.8423


R9 + hr x yr                                            589      0.9107  0.9125   0.9063      0.8967



carried-forward model, chronological split by gradient descent: R2 0.9061 (status converged, 15457 iterations); closed form 0.9063
seeded validation 0.9107  vs  chronological 0.9061: gap +0.0045


## 9. Learning curves and residuals (carried-forward model)

Learning curves use closed-form fits on random subsets of the training part (reproducible with the team seed).

In [11]:
rng = np.random.default_rng(SEED)
fracs = np.linspace(0.1, 1.0, 10)
(Xa0, Xv0), _, _, _ = design(train_set, [val_set], {'degrees': {}})
curves = {}
for lab, (XA, XV) in [('R0 linear', (Xa0, Xv0)), (pick['rung'][:3].strip() + ' carried forward', (Xa_c, Xv_c))]:
    trc, vac = [], []
    for fr in fracs:
        idx = rng.choice(len(XA), size=int(fr * len(XA)), replace=False)
        w_, b_ = ols(XA[idx], yt[idx])
        trc.append(core.r2_score_np(yt[idx], core.predict(XA[idx], w_, b_))); vac.append(core.r2_score_np(yv, core.predict(XV, w_, b_)))
    curves[lab] = (trc, vac)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (lab, (trc, vac)) in zip(axes, curves.items()):
    ax.plot(fracs * len(Xa0), trc, 'b-o', label='train'); ax.plot(fracs * len(Xa0), vac, 'r-s', label='validation')
    ax.set_title(f'Learning curve: {lab}'); ax.set_xlabel('training rows'); ax.set_ylabel('R2'); ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); fig.savefig(PLOTS / '03_learning_curves.png', dpi=140); plt.close(fig)
for lab, (trc, vac) in curves.items(): print(f'{lab:<28} final gap {trc[-1]-vac[-1]:+.4f}; gap at 10% of the data {trc[0]-vac[0]:+.4f}')

resid = yv - core.predict(Xv_c, W_C, B_C)
res_df = val_set[['hr', 'workingday']].copy(); res_df['residual'] = resid
hb = res_df.groupby(['hr', 'workingday'])['residual'].mean().unstack()
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(core.predict(Xv_c, W_C, B_C), resid, alpha=0.3, s=10, color='teal'); axes[0].axhline(0, color='red', ls='--')
axes[0].set_title('Residuals vs fitted (validation)'); axes[0].set_xlabel('fitted cnt'); axes[0].set_ylabel('residual'); axes[0].grid(alpha=0.3)
hb.plot(ax=axes[1], marker='o'); axes[1].axhline(0, color='k', ls=':'); axes[1].set_title('Mean residual by hour x workingday'); axes[1].set_xlabel('hour'); axes[1].grid(alpha=0.3)
fig.tight_layout(); fig.savefig(PLOTS / '03_residuals.png', dpi=140); plt.close(fig)
print(f'validation residual: mean {resid.mean():+.2f}, RMSE {np.sqrt((resid**2).mean()):.2f}; largest |mean residual| by hour x workingday: {np.abs(hb.values).max():.1f}')
print('[SAVED] plots/03_learning_curves.png, plots/03_residuals.png')

R0 linear                    final gap +0.0199; gap at 10% of the data +0.0425
R9 carried forward           final gap +0.0170; gap at 10% of the data +0.4156
validation residual: mean -0.43, RMSE 52.92; largest |mean residual| by hour x workingday: 38.4
[SAVED] plots/03_learning_curves.png, plots/03_residuals.png


## 10. Artifact for Phase 4

In [12]:
extra = {
    'phase2_val_r2': META_P2['val_r2'],
    'chosen_rung': pick['rung'], 'selection_tolerance': TOL_SEL, 'selection_score': 'day-grouped CV R2 inside the training part',
    'trend_powers_dropped': True,
    'train_r2': pick['train_r2'], 'val_r2': pick['val_r2'], 'val_rmse': pick['val_rmse'],
    'cv_r2': pick['cv_r2'], 'cv_std': pick['cv_std'],
    'chrono_r2_closed_form': pick['chrono_r2'], 'chrono_r2_gd': CH_GD,
    'chrono_cut_date': str(chrono_tr['dteday'].max()),
    'test_like_holdout_r2': test_like[pick['rung']],
    'gd_iterations': pick['iters'], 'gd_status': pick['status'],
    'ladder': [{k: (float(v) if isinstance(v, (np.floating, float)) else v) for k, v in r.items()} for r in rows],
}
P3_SHA = p3.save_artifacts(P3_STEM, W_C, B_C, NAMES_C, PP_C, META_P2, SPEC, p1_train_idx, p1_val_idx, extra=extra)

w_rt, b_rt, meta_rt = p3.load_artifacts(P3_STEM)
assert np.array_equal(w_rt, W_C) and b_rt == B_C and meta_rt['phase3_w_sha256'] == P3_SHA
assert meta_rt['expanded_feature_names'] == NAMES_C and len(w_rt) == len(NAMES_C)
(_, Xv_rt), _, _, _ = design(train_set, [val_set], meta_rt['complexity_spec'])
assert abs(core.r2_score_np(yv, core.predict(Xv_rt, w_rt, b_rt)) - pick['val_r2']) < 1e-12
print(f'[PASS] artifact round trip; SHA-256(w) = {P3_SHA}; {len(w_rt)} features')

[PASS] artifact round trip; SHA-256(w) = fc9a3ab8de1a7cd5a2fe6df59b37d74808ce126ded15822795bf9840f492dbfb; 589 features


In [13]:
# Numbers of this run, in one place
print(f'seed {SEED}; train/val rows {len(train_set)}/{len(val_set)}')
print(f'Phase 2 model: {len(P2_FEATS)} features, val R2 {rows[1]["val_r2"]:.4f}, chronological R2 {rows[1]["chrono_r2"]:.4f}')
print(f'carried forward: {pick["rung"]}')
print(f'  {pick["p"]} features | train R2 {pick["train_r2"]:.4f} | val R2 {pick["val_r2"]:.4f} (RMSE {pick["val_rmse"]:.2f}) | CV R2 {pick["cv_r2"]:.4f} +-{pick["cv_std"]:.4f} | chronological R2 {CH_GD:.4f} | days 18-19 R2 {test_like[pick["rung"]]:.4f}')
print(f'  GD: {pick["iters"]} iterations, status {pick["status"]}')
print(f'total runtime {time.time()-T0:.0f}s')

seed 42648; train/val rows 8708/2178
Phase 2 model: 62 features, val R2 0.6858, chronological R2 0.5099
carried forward: R9 + hr x yr
  589 features | train R2 0.9272 | val R2 0.9107 (RMSE 52.92) | CV R2 0.9125 +-0.0058 | chronological R2 0.9061 | days 18-19 R2 0.8967
  GD: 13924 iterations, status converged
total runtime 227s


## Outcome